<a href="https://colab.research.google.com/github/leticiasdrummond/Modelos-Base/blob/main/Modelagem_Sistemas_de_Energia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Olá! Como seu Parceiro de Programação, estou pronto para continuarmos a nossa jornada pelo mundo da otimização! 🚀

Analisar múltiplos artigos e traduzi-los para código é um excelente exercício. Como você enviou vários anexos que abordam problemas com estruturas semelhantes (como o uso do HOMER para dimensionamento), selecionei **três dos documentos mais focados em modelagem matemática e otimização** para detalhar minuciosamente os 5 passos que você solicitou.


---

In [5]:
!pip install gurobipy




### 1. Artigo SBPO 2022 (Caio dos Santos et al.)

**1) Referência ABNT**
SANTOS, C. et al. Multiobjective EV Charging Station Planning for Large-Scale Transportation Systems. In: LIV Simpósio Brasileiro de Pesquisa Operacional (SBPO), 2022, Juiz de Fora. **Anais...** Juiz de Fora: SOBRAPO, 2022.

**2) Declaração do Problema (Problem Statement)**
"Precisamos decidir onde instalar Estações de Carregamento (EC) para Veículos Elétricos em uma rede de transportes de grande escala (como rodovias) e qual deve ser o tamanho do sistema de painéis solares (PV) em cada uma. O objetivo é minimizar os custos totais de investimento (construção) e operação (compra de energia), garantindo que a demanda de recarga dos motoristas seja atendida e aproveitando o sistema de compensação de créditos de energia (net-metering)."

**3) Componentes Fundamentais**

* **Variáveis de Decisão:** Quais locais receberão as estações (variável binária: constrói ou não constrói) e qual a capacidade instalada de painéis solares em cada local (variável contínua).
* **Função Objetivo:** Minimizar o Custo Total (Custo de Instalação das Estações + Custo de Instalação PV + Custo de Operação/Compra de energia da rede).
* **Restrições:** Toda a demanda de recarga de veículos mapeada na região deve ser atendida; a área máxima de instalação de painéis solares (sobre os estacionamentos/carports) deve ser respeitada; balanço de energia (energia gerada + comprada - exportada = consumida).
* **Dados:** Dados de tráfego, custos de instalação, tarifas de energia e previsão de geração fotovoltaica.

**4) Modelo Matemático (Mathematical terms)**

* **Sets:** $i \in I$ (Locais candidatos).
* **Parameters:** $D_i$ (Demanda no local $i$), $C^{EC}_i$ (Custo fixo da estação), $C^{PV}$ (Custo por kW de PV), $A_i$ (Área máxima PV), $Tarifa$ (Custo da energia).
* **Variables:** $x_i \in \{0, 1\}$ (1 se construir estação em $i$), $S_i \ge 0$ (Tamanho do PV em kW), $P^{grid}_i \ge 0$ (Energia importada).
* **Objective:** $\text{Min} \sum (C^{EC}_i \cdot x_i + C^{PV} \cdot S_i + Tarifa \cdot P^{grid}_i)$
* **Constraints:**
* $P^{grid}_i + S_i \cdot (\text{fator de sol}) \ge D_i \cdot x_i$ (Balanço de energia)
* $S_i \le A_i \cdot x_i$ (Só pode instalar PV se a estação for construída, e até o limite de área).



**5) Tradução para Código**

In [9]:
import gurobipy as gp
from gurobipy import GRB

def planejar_estacoes_sbpo():
    """Modelo de Alocação de Estações de Recarga com PV"""
    locais = [1, 2, 3] # 3 locais candidatos na rodovia
    demanda = {1: 100, 2: 250, 3: 150} # kW necessários
    custo_fixo_ec = {1: 50000, 2: 60000, 3: 55000} # R$ para construir
    area_max = {1: 50, 2: 100, 3: 80} # kW máximos de PV que cabem no teto

    custo_pv_kw = 3000
    tarifa_energia = 0.50 # R$/kWh

    modelo = gp.Model("Planejamento_EC_PV")

    # Variáveis
    x = modelo.addVars(locais, vtype=GRB.BINARY, name="Construir_EC")
    s_pv = modelo.addVars(locais, lb=0, name="Tamanho_PV")
    p_grid = modelo.addVars(locais, lb=0, name="Compra_Rede")

    # Função Objetivo: Minimizar custos de investimento + operação
    custo_inv = gp.quicksum(x[i] * custo_fixo_ec[i] + s_pv[i] * custo_pv_kw for i in locais)
    custo_op = gp.quicksum(p_grid[i] * tarifa_energia for i in locais)
    modelo.setObjective(custo_inv + custo_op, GRB.MINIMIZE)

    # Restrições
    for i in locais:
        # Só atende a demanda se a estação for construída (simplificação)
        modelo.addConstr(p_grid[i] + s_pv[i] >= demanda[i] * x[i], name=f"Demanda_{i}")

        # Limite de área e dependência de construção
        modelo.addConstr(s_pv[i] <= area_max[i] * x[i], name=f"Area_{i}")

    # Exigir que pelo menos 400 kW de demanda total sejam cobertos (Exemplo de meta)
    modelo.addConstr(gp.quicksum(demanda[i] * x[i] for i in locais) >= 400, name="Meta_Cobertura")

    modelo.optimize()

    if modelo.status == GRB.OPTIMAL:
        print("\n--- Resultados do Modelo SBPO 2022 ---")
        print(f"Custo Total Ótimo: R$ {modelo.objVal:,.2f}")
        for i in locais:
            if x[i].X > 0.5: # Se a estação for construída
                print(f"Local {i}: Estação construída. Tamanho PV: {s_pv[i].X:.2f} kW. Energia da Rede: {p_grid[i].X:.2f} kW")
            else:
                print(f"Local {i}: Estação NÃO construída.")
    else:
        print("Nenhuma solução ótima encontrada para o modelo SBPO 2022.")
3

In [15]:
planejar_estacoes_sbpo()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 7 rows, 9 columns and 18 nonzeros (Min)
Model fingerprint: 0x765d0940
Model has 9 linear objective coefficients
Variable types: 6 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [5e-01, 6e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [4e+02, 4e+02]

Presolve removed 7 rows and 9 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 2 available processors)

Solution count 1: 115200 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.152000000000e+05, best bound 1.152000000000e+05, gap 0.0000%

--- Resultados do Modelo SBPO 2022 ---
Custo Tot

In [12]:
dimensionar_microrrede_ufsm()

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 8 rows, 18 columns and 22 nonzeros (Min)
Model fingerprint: 0xf93c6c39
Model has 6 linear objective coefficients
Coefficient statistics:
  Matrix range     [5e-01, 1e+00]
  Objective range  [6e-01, 1e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 5e+01]

Presolve removed 8 rows and 18 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00

--- Resultados do Modelo TCC UFSM 2020 ---
Custo Total Mínimo (NPC Anualizado

In [16]:
gestao_energia_utfpr()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 48 rows, 96 columns and 167 nonzeros (Min)
Model fingerprint: 0xf1964d37
Model has 24 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e-01, 1e+00]
  Bounds range     [2e+01, 1e+02]
  RHS range        [2e+01, 1e+02]

Presolve removed 27 rows and 53 columns
Presolve time: 0.01s
Presolved: 21 rows, 43 columns, 63 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    7.7200000e+02   5.525000e+02   0.000000e+00      0s
      19    1.0560000e+03   0.000000e+00   0.000000e+00      0s

Solved in 19 iterations and 0.02 seconds (0.00 work units)
Optimal objective  1.056000000e+03

--- Resultados do Modelo Tese Doutorado UTFPR 2021 ---
Custo da Fatura d

---

### 2. TCC UFSM 2020 (Pedro Augusto de Almeida Real)

**1) Referência ABNT**
REAL, P. A. A. **Análise de modelo de microrrede de energia solar conectada à rede para carregamento de veículos elétricos**. 2020. 57 f. Trabalho de Conclusão de Curso (Bacharelado em Engenharia Elétrica) – Universidade Federal de Santa Maria, Santa Maria, 2020.

**2) Declaração do Problema (Problem Statement)**
"Temos uma estação de carregamento de veículos elétricos conectada à rede. Queremos saber qual é a combinação ideal (dimensionamento) de Painéis Solares e Baterias que devemos comprar para **minimizar o Custo Presente Líquido (NPC)** do projeto ao longo de sua vida útil, garantindo que toda a carga dos veículos seja perfeitamente atendida hora a hora, levando em conta os dados reais de irradiação solar e consumo."

**3) Componentes Fundamentais**

* **Variáveis de Decisão:** Capacidade instalada do sistema Fotovoltaico (kW) e número de baterias de armazenamento.
* **Função Objetivo:** Minimizar o NPC (Custo capital de componentes + custos de reposição + operação e manutenção - valor residual).
* **Restrições:** A energia fornecida aos veículos elétricos deve igualar a demanda exigida em todas as horas do ano (8760 horas); limites de carga e descarga das baterias.
* **Dados:** Perfil de irradiação solar (ex: dados do INMET), custos dos equipamentos e perfil de carga dos VEs.

**4) Modelo Matemático (Mathematical terms)**

* **Sets:** $t \in T$ (Horas do ano, 1 a 8760).
* **Parameters:** $C_{PV}, C_{Bat}$ (Custos anualizados), $D_t$ (Demanda VEs).
* **Variables:** $Cap_{PV}$ e $Cap_{Bat}$ (Capacidades a instalar). $P_{grid, t}$ (Uso da rede), $P_{ch, t}, P_{dis, t}$ (Bateria).
* **Objective:** $\text{Min} \ (C_{PV} \cdot Cap_{PV}) + (C_{Bat} \cdot Cap_{Bat}) + \sum (C_{energia} \cdot P_{grid, t})$
* **Constraints:** $P_{grid, t} + (Cap_{PV} \cdot Rad_t) + P_{dis, t} - P_{ch, t} = D_t$

**5) Tradução para Código**
*Nota: Este problema simula exatamente o motor matemático por trás de softwares como o HOMER Energy, que foi usado no TCC.*

In [10]:
import gurobipy as gp
from gurobipy import GRB

def dimensionar_microrrede_ufsm():
    """Dimensionamento de Microrrede (PV + Bateria) conectada à rede"""
    T = [1, 2, 3, 4] # Representando 4 períodos para fins didáticos
    demanda_ve = {1: 10, 2: 50, 3: 40, 4: 10} # kW
    rad_solar = {1: 0, 2: 0.8, 3: 0.5, 4: 0}
    custo_rede = 0.60

    modelo = gp.Model("Microgrid_UFSM")

    # Variáveis (Dimensionamento)
    cap_pv = modelo.addVar(name="Capacidade_PV", lb=0)
    cap_bat = modelo.addVar(name="Capacidade_Bateria", lb=0)

    # Variáveis (Operação Operação - Despacho)
    p_grid = modelo.addVars(T, name="Grid")
    p_ch = modelo.addVars(T, name="Bat_Carga")
    p_dis = modelo.addVars(T, name="Bat_Descarga")
    soc = modelo.addVars(T, name="StateOfCharge")

    # Objetivo: Minimizar investimento + custo da rede elétrica
    # Valores fictícios de custo anualizado
    modelo.setObjective(1200 * cap_pv + 800 * cap_bat + gp.quicksum(p_grid[t]*custo_rede for t in T), GRB.MINIMIZE)

    # Restrições de Balanço de Potência
    for t in T:
        modelo.addConstr(p_grid[t] + cap_pv * rad_solar[t] + p_dis[t] - p_ch[t] == demanda_ve[t])
        modelo.addConstr(soc[t] <= cap_bat) # Bateria não pode carregar mais que seu tamanho

    modelo.optimize()

    if modelo.status == GRB.OPTIMAL:
        print("\n--- Resultados do Modelo TCC UFSM 2020 ---")
        print(f"Custo Total Mínimo (NPC Anualizado Fictício): R$ {modelo.objVal:,.2f}")
        print(f"Capacidade PV Instalada: {cap_pv.X:.2f} kW")
        print(f"Capacidade de Bateria Instalada: {cap_bat.X:.2f} kWh")
        print("\nOperação Horária (Exemplo para os 4 períodos):")
        for t in T:
            print(f"Período {t}: Rede={p_grid[t].X:.2f} kW, Carga Bat={p_ch[t].X:.2f} kW, Descarga Bat={p_dis[t].X:.2f} kW, SOC={soc[t].X:.2f} kWh")
    else:
        print("Nenhuma solução ótima encontrada para o modelo TCC UFSM 2020.")


---

### 3. Tese de Doutorado UTFPR 2021 (Juliana D'Angela Mariano)

**1) Referência ABNT**
MARIANO, J. D. **A integração dos sistemas de armazenamento de energia nos sistemas fotovoltaicos: estudo de caso da gestão da energia na UTFPR**. 2021. Tese (Doutorado em Engenharia Civil) – Universidade Tecnológica Federal do Paraná, Curitiba, 2021.

**2) Declaração do Problema (Problem Statement)**
"Precisamos gerenciar o consumo de energia dos prédios da universidade (UTFPR) integrando Sistemas Fotovoltaicos com Baterias. O objetivo é **minimizar a fatura de energia elétrica** do campus, utilizando baterias para armazenar o excesso de geração solar durante o dia e descarregá-la nos horários de ponta (onde a tarifa da concessionária é muito mais cara), respeitando a curva de demanda do campus."

**3) Componentes Fundamentais**

* **Variáveis de Decisão:** O regime de despacho da bateria a cada hora (quanto carregar da rede/solar, quanto descarregar para o prédio).
* **Função Objetivo:** Minimizar o custo total com a conta de luz (Tarifa Branca/Horária x Consumo Líquido).
* **Restrições:** As regras tarifárias da concessionária, os limites técnicos do inversor fotovoltaico e da profundidade de descarga da bateria (para não estragar a vida útil do equipamento).
* **Dados:** Demanda média horária do campus (Curva de carga), perfil de geração diário, tarifas de ponta e fora de ponta.

**4) Modelo Matemático (Mathematical terms)**

* **Parameters:** $Tarifa_t$ (Alta no horário de ponta, baixa fora de ponta), $D_t$ (Demanda do Prédio), $G_{FV, t}$ (Geração PV real).
* **Variables:** $E^{rede}_t$ (Energia comprada da concessionária), $B^{dis}_t$ (Descarga da bateria).
* **Objective:** $\text{Min} \sum (E^{rede}_t \cdot Tarifa_t)$
* **Constraints:** * Balanço do Prédio: $E^{rede}_t + G_{FV, t} + B^{dis}_t - B^{ch}_t = D_t$
* Armazenamento: Bateria só pode ser descarregada até uma Profundidade de Descarga (DoD) de 80%.



**5) Tradução para Código**

In [11]:
import gurobipy as gp
from gurobipy import GRB

def gestao_energia_utfpr():
    """Gestão de Energia: Arbitragem Tarifária (Peak Shaving)"""
    T = range(1, 25) # 24 horas do dia

    # Dados didáticos: Tarifa muito alta entre 18h e 21h (Horário de Ponta)
    tarifa = {t: 1.20 if 18 <= t <= 21 else 0.40 for t in T}
    demanda = {t: 100 for t in T} # Consumo constante de 100 kW do prédio
    geracao_pv = {t: 50 if 9 <= t <= 16 else 0 for t in T} # Sol das 9h às 16h

    modelo = gp.Model("Peak_Shaving_UTFPR")

    # Variáveis Operacionais (O sistema já está instalado)
    compra_rede = modelo.addVars(T, lb=0, name="Rede")
    bat_ch = modelo.addVars(T, lb=0, name="Carrega_Bat")
    bat_dis = modelo.addVars(T, lb=0, ub=30, name="Descarrega_Bat") # Inversor max 30kW
    soc = modelo.addVars(T, lb=20, ub=100, name="Carga_Bateria") # Limite DoD de 80% (mínimo 20)

    # Objetivo: Minimizar a conta de luz
    modelo.setObjective(gp.quicksum(compra_rede[t] * tarifa[t] for t in T), GRB.MINIMIZE)

    # Restrições Dinâmicas
    for t in T:
        # A energia que entra no prédio deve atender a demanda
        modelo.addConstr(compra_rede[t] + geracao_pv[t] + bat_dis[t] - bat_ch[t] == demanda[t])

        # Lógica da bateria
        if t == 1:
            modelo.addConstr(soc[t] == 20 + bat_ch[t] - bat_dis[t]) # Estado inicial
        else:
            modelo.addConstr(soc[t] == soc[t-1] + bat_ch[t] - bat_dis[t])

    modelo.optimize()

    if modelo.status == GRB.OPTIMAL:
        print("\n--- Resultados do Modelo Tese Doutorado UTFPR 2021 ---")
        print(f"Custo da Fatura de Energia Mínimo: R$ {modelo.objVal:,.2f}")
        print("\nOperação Horária:")
        for t in T:
            print(f"Hora {t:02d}: Compra Rede={compra_rede[t].X:.2f} kW, Carga Bat={bat_ch[t].X:.2f} kW, Descarga Bat={bat_dis[t].X:.2f} kW, SOC={soc[t].X:.2f} kWh")
    else:
        print("Nenhuma solução ótima encontrada para o modelo Tese Doutorado UTFPR 2021.")
